In [4]:
import pandas as pd
import numpy as np  # Used for weighted average and sign function

# 1. Load data
parent = pd.read_csv('parent_order.csv')
child = pd.read_csv('child_order.csv')
trade = pd.read_csv('trade.csv')
quote = pd.read_csv('quote.csv')

# Combine date and time, then sort
for df in [child, trade, quote]:
    df['datetime'] = pd.to_datetime(df['date'] + ' ' + df['time'])
    df.sort_values('datetime', inplace=True)

results = []

for _, p_order in parent.iterrows():
    order_id, sym, date, side = p_order['orderid'], p_order['sym'], p_order['date'], p_order['side']

    # Order start and end times
    start_time = pd.to_datetime(f"{date} {p_order['starttime']}")
    end_time = pd.to_datetime(f"{date} {p_order['endtime']}")

    # Filter data for the current order
    c_order = child[child['parentid'] == order_id]
    m_trade = trade[(trade['sym'] == sym) & (trade['date'] == date)].copy()
    m_quote = quote[(quote['sym'] == sym) & (quote['date'] == date)]

    # Handle late trading: normalize time after 14:57 to 15:00:00
    m_trade.loc[m_trade['datetime'].dt.time >= pd.to_datetime('14:57:00').time(), 'datetime'] = pd.to_datetime(f"{date} 15:00:00")
    m_trade.sort_values('datetime', inplace=True)

    # ---------------- Core Metrics ----------------
    total_qty = c_order['size'].sum()
    notional = (c_order['price'] * c_order['size']).sum()
    avg_price = notional / total_qty

    # ADV: Percentage of total daily market volume
    adv_pct = total_qty / m_trade['size'].sum()

    interval_trades = m_trade[(m_trade['datetime'] >= start_time) & (m_trade['datetime'] <= end_time)]
    # Trading Speed: Percentage of volume during the active order window
    speed = total_qty / interval_trades['size'].sum()

    interval_quotes = m_quote[(m_quote['datetime'] >= start_time) & (m_quote['datetime'] <= end_time)]
    spread_bps = ((interval_quotes['ask'] - interval_quotes['bid']) / ((interval_quotes['ask'] + interval_quotes['bid']) / 2) * 10000).mean()

    # ---------------- Benchmark Prices ----------------
    open_price = m_trade.iloc[0]['price']
    close_price = m_trade.iloc[-1]['price']

    # Arrival: Use open price if before market open, else use latest mid-price
    if start_time.time() <= pd.to_datetime('09:30:00').time():
        arrival_price = open_price
    else:
        arrival_q = m_quote[m_quote['datetime'] <= start_time]
        arrival_price = (arrival_q.iloc[-1]['bid'] + arrival_q.iloc[-1]['ask']) / 2 if not arrival_q.empty else open_price

    # iVWAP
    ivwap_price = (interval_trades['price'] * interval_trades['size']).sum() / interval_trades['size'].sum()

    # PWP5: Simulate execution at a 5% participation rate
    pwp_trades = m_trade[m_trade['datetime'] >= start_time].copy()
    pwp_trades['sim_qty'] = pwp_trades['size'] * 0.05
    pwp_trades['cum_sim_qty'] = pwp_trades['sim_qty'].cumsum()

    overshoot_idx = pwp_trades[pwp_trades['cum_sim_qty'] > total_qty].index
    if not overshoot_idx.empty:
        first_overshoot = overshoot_idx[0]
        # Truncate excess volume
        pwp_trades.loc[first_overshoot, 'sim_qty'] -= (pwp_trades.loc[first_overshoot, 'cum_sim_qty'] - total_qty)
        pwp_trades = pwp_trades.loc[:first_overshoot]

    pwps_price = (pwp_trades['price'] * pwp_trades['sim_qty']).sum() / total_qty

    # ---------------- Cost and Fill Statistics ----------------
    # Formula: (Benchmark - Exec Price). Positive value = profit/savings
    calc_cost = lambda bench_p: 10000.0 * side * (bench_p - avg_price) / bench_p

    moo_pct = c_order[c_order['datetime'].dt.time == pd.to_datetime('09:25:00').time()]['size'].sum() / total_qty
    moc_pct = c_order[c_order['datetime'].dt.time >= pd.to_datetime('14:57:00').time()]['size'].sum() / total_qty

    # Calculate passive/aggressive fills using mid-price
    c_order_q = pd.merge_asof(c_order, m_quote[['datetime', 'bid', 'ask']], on='datetime', direction='backward')
    passive_qty = aggressive_qty = 0

    for _, row in c_order_q.iterrows():
        p, b, a, sz = row['price'], row['bid'], row['ask'], row['size']
        mid = (b + a) / 2 if pd.notna(b) else open_price

        # Tag: 1 is passive, -1 is aggressive
        tag = side * np.sign(mid - p)
        if tag == 1:
            passive_qty += sz
        elif tag == -1:
            aggressive_qty += sz

    results.append({
        'OrderID': order_id,
        'Notional (Million CNY)': notional / 1_000_000,
        'ADV%': adv_pct,
        'Trading Speed': speed,
        'Spread (bps)': spread_bps,
        'Open': calc_cost(open_price),
        'Arrival': calc_cost(arrival_price),
        'iVWAP': calc_cost(ivwap_price),
        'Close': calc_cost(close_price),
        'PWP5': calc_cost(pwps_price),
        'MOO%': moo_pct,
        'MOC%': moc_pct,
        'Passive%': passive_qty / total_qty,
        'Aggressive%': aggressive_qty / total_qty
    })

# 2. Calculate 'All' summary row and format output
df = pd.DataFrame(results)
weights = df['Notional (Million CNY)']

# Add 'All' row
all_row = {'OrderID': 'All', 'Notional (Million CNY)': weights.sum()}
cols_to_avg = ['ADV%', 'Trading Speed', 'Spread (bps)', 'Open', 'Arrival',
               'iVWAP', 'Close', 'PWP5', 'MOO%', 'MOC%', 'Passive%', 'Aggressive%']

for col in cols_to_avg:
    all_row[col] = np.average(df[col], weights=weights)

df = pd.concat([df, pd.DataFrame([all_row])], ignore_index=True)

# Round numbers to 2 decimal places
for col in ['Notional (Million CNY)', 'Spread (bps)', 'Open', 'Arrival', 'iVWAP', 'Close', 'PWP5']:
    df[col] = df[col].round(2)
for col in ['ADV%', 'Trading Speed', 'MOO%', 'MOC%', 'Passive%', 'Aggressive%']:
    df[col] = (df[col] * 100).round(2)

df.to_csv('Assignment_2_Output.csv', index=False)
print(df.to_string(index=False))

OrderID  Notional (Million CNY)  ADV%  Trading Speed  Spread (bps)    Open  Arrival  iVWAP   Close   PWP5  MOO%  MOC%  Passive%  Aggressive%
   V001                   15.89  0.90           1.96          8.60  -30.88   -30.88  -2.53  121.47   0.76  3.19  0.00     32.29        64.53
   V002                   23.39  0.79           1.60          3.98    8.15    25.83  -4.19  -33.89 -13.14  0.00  0.00     32.85        66.69
   V003                    4.47  0.35           1.09          7.32  -69.38   -69.38  -6.43 -112.44 -21.57  9.94  0.00     48.27        41.79
   V004                   28.49  1.82           1.82          4.06    3.77     3.77  -1.66  -10.11   4.96  0.00  3.27     43.32        52.20
   V005                   33.14  2.27           2.27          7.16 -149.28  -149.28  17.86 -117.69  93.13  0.73  3.00     62.38        36.87
    All                  105.39  1.53           1.90          5.84  -51.71   -47.79   3.58  -33.71  26.91  1.13  1.83     45.53        52.02
